In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../dataset/amazonsales.csv')
print(f'Raw data shape: {df.shape}')
print(df.columns.tolist())

Raw data shape: (1465, 16)
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


In [2]:
def clean_price(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace('[₹,]', '', regex=True)
              .str.strip(),
        errors='coerce'
    )

price_cols = ['discounted_price', 'actual_price']
for col in price_cols:
    if col in df.columns:
        df[col] = clean_price(df[col])
        print(f'{col} → min: {df[col].min():.0f}, max: {df[col].max():.0f}')

discounted_price → min: 39, max: 77990
actual_price → min: 39, max: 139900


In [3]:
if 'discount_percentage' in df.columns:
    df['discount_percentage'] = pd.to_numeric(
        df['discount_percentage'].astype(str)
                                 .str.replace('%', '', regex=False)
                                 .str.strip(),
        errors='coerce'
    )
    print(f'discount_percentage → min: {df["discount_percentage"].min()}, max: {df["discount_percentage"].max()}')

discount_percentage → min: 0, max: 94


In [4]:
if 'rating' in df.columns:
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)].shape[0]
    print(f'Invalid ratings (outside 1-5): {invalid_ratings}')
    df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'rating'] = np.nan

Invalid ratings (outside 1-5): 0


In [11]:
if 'rating_count' in df.columns:
    df['rating_count'] = pd.to_numeric(
        df['rating_count'].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )
    before = df['rating_count'].isnull().sum()
    df['rating_count'] = df['rating_count'].fillna(df['rating_count'].median())
    print(f'rating_count nulls fixed: {before} → {df["rating_count"].isnull().sum()}')

rating_count nulls fixed: 2 → 0


In [6]:
text_cols = ['product_name', 'category', 'about_product']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        print(f'{col} cleaned — sample: {df[col].iloc[0][:60]}')

product_name cleaned — sample: Wayona Nylon Braided USB to Lightning Fast Charging and Data
category cleaned — sample: Computers&Accessories|Accessories&Peripherals|Cables&Accesso
about_product cleaned — sample: High Compatibility : Compatible With iPhone 12, 11, X/XsMax/


In [7]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]
print(f'Duplicates removed: {before - after} rows')
print(f'Clean dataset shape: {df.shape}')

Duplicates removed: 0 rows
Clean dataset shape: (1465, 16)


In [8]:
if 'actual_price' in df.columns and 'discounted_price' in df.columns:
    df['savings_amount'] = df['actual_price'] - df['discounted_price']
    df['savings_pct'] = ((df['savings_amount'] / df['actual_price']) * 100).round(2)
    print('Derived columns added: savings_amount, savings_pct')
    print(df[['actual_price','discounted_price','savings_amount','savings_pct']].head(3))

Derived columns added: savings_amount, savings_pct
   actual_price  discounted_price  savings_amount  savings_pct
0        1099.0             399.0           700.0        63.69
1         349.0             199.0           150.0        42.98
2        1899.0             199.0          1700.0        89.52


In [9]:
print('=== FINAL NULL COUNTS ===')
print(df.isnull().sum())
print('\n=== FINAL DATA TYPES ===')
print(df.dtypes)
print('\n=== FINAL SHAPE ===')
print(df.shape)

=== FINAL NULL COUNTS ===
product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 1
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
savings_amount         0
savings_pct            0
dtype: int64

=== FINAL DATA TYPES ===
product_id                 str
product_name               str
category                   str
discounted_price       float64
actual_price           float64
discount_percentage      int64
rating                 float64
rating_count           float64
about_product              str
user_id                    str
user_name                  str
review_id                  str
review_title               str
review_content             str
img_link                   str
product_link        

In [10]:
df.to_csv('../dataset/amazon_sales_cleaned.csv', index=False, encoding='utf-8-sig')
print('Saved → amazon_sales_cleaned.csv')
print(f'Final shape: {df.shape}')

Saved → amazon_sales_cleaned.csv
Final shape: (1465, 18)
